<a href="https://colab.research.google.com/github/adinuansah/people-attrition-prediction/blob/main/notebooks/00_generate_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd

# Reproducible random number generator
rng = np.random.default_rng(seed=42)

# Number of fictional employees
N_EMPLOYEES = 2000

# Historical observation period
START_DATE = pd.Timestamp("2022-01-01")
END_DATE = pd.Timestamp("2025-12-31")

# Organizational structure
DEPARTMENTS = [
    "Engineering",
    "Product",
    "Design",
    "Data",
    "Operations",
    "Finance",
    "People"
]

# Approximate department distribution
DEPARTMENT_PROBABILITIES = [
    0.35,
    0.15,
    0.10,
    0.12,
    0.13,
    0.08,
    0.07
]

print("Employees:", N_EMPLOYEES)
print("Departments:", len(DEPARTMENTS))
print("Probabilities sum:",
      sum(DEPARTMENT_PROBABILITIES))
print("Observation period:", START_DATE.date(),
      "to", END_DATE.date())


Employees: 2000
Departments: 7
Probabilities sum: 1.0
Observation period: 2022-01-01 to 2025-12-31


In [2]:

# Step 2A — Employee IDs and hire dates

employee_ids = [
    f"E{i:05d}"
    for i in range(1, N_EMPLOYEES + 1)
]

# Allow some employees to have joined before
# the historical observation period.
HIRE_START_DATE = pd.Timestamp("2019-01-01")
HIRE_END_DATE = pd.Timestamp("2025-09-30")

hire_dates = HIRE_START_DATE + pd.to_timedelta(
    rng.integers(
        low=0,
        high=(HIRE_END_DATE - HIRE_START_DATE).days + 1,
        size=N_EMPLOYEES
    ),
    unit="D"
)

print("First five employee IDs:", employee_ids[:5])
print("First five hire dates:", hire_dates[:5])


First five employee IDs: ['E00001', 'E00002', 'E00003', 'E00004', 'E00005']
First five hire dates: DatetimeIndex(['2019-08-09', '2024-03-22', '2023-06-02', '2021-12-17',
               '2021-12-03'],
              dtype='datetime64[ns]', freq=None)


In [3]:

# Step 2B — Departments and job levels

initial_departments = rng.choice(
    DEPARTMENTS,
    size=N_EMPLOYEES,
    p=DEPARTMENT_PROBABILITIES
)

JOB_LEVELS = ["L1", "L2", "L3", "L4", "L5", "L6"]

JOB_LEVEL_PROBABILITIES = [
    0.10, 0.25, 0.30, 0.20, 0.10, 0.05
]

initial_job_levels = rng.choice(
    JOB_LEVELS,
    size=N_EMPLOYEES,
    p=JOB_LEVEL_PROBABILITIES
)

print("Sample departments:", initial_departments[:5])
print("Sample job levels:", initial_job_levels[:5])


Sample departments: ['Engineering' 'Product' 'Engineering' 'Engineering' 'Data']
Sample job levels: ['L4' 'L3' 'L2' 'L4' 'L1']


In [4]:

# Step 2C — Build the employee master table

employees = pd.DataFrame({
    "employee_id": employee_ids,
    "hire_date": hire_dates,
    "initial_department": initial_departments,
    "initial_job_level": initial_job_levels
})

display(employees.head(10))

print("Dataset shape:", employees.shape)


,employee_id,hire_date,initial_department,initial_job_level
0,E00001,2019-08-09,Engineering,L4
1,E00002,2024-03-22,Product,L3
2,E00003,2023-06-02,Engineering,L2
3,E00004,2021-12-17,Engineering,L4
4,E00005,2021-12-03,Data,L1
5,E00006,2024-10-17,Product,L4
6,E00007,2019-07-31,Finance,L3
7,E00008,2023-09-16,Engineering,L2
8,E00009,2020-05-11,Operations,L1
9,E00010,2019-08-21,Product,L4


Dataset shape: (2000, 4)


In [5]:

# Step 2D — Employee master quality checks

quality_checks = {
    "row_count_is_2000":
        len(employees) == N_EMPLOYEES,

    "employee_ids_are_unique":
        employees["employee_id"].is_unique,

    "no_missing_values":
        not employees.isna().any().any(),

    "hire_dates_in_range":
        employees["hire_date"].between(
            HIRE_START_DATE,
            HIRE_END_DATE
        ).all(),

    "valid_departments":
        employees["initial_department"].isin(
            DEPARTMENTS
        ).all(),

    "valid_job_levels":
        employees["initial_job_level"].isin(
            JOB_LEVELS
        ).all()
}

for check, passed in quality_checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

assert all(quality_checks.values()), (
    "Employee master failed data-quality checks."
)


row_count_is_2000: PASS
employee_ids_are_unique: PASS
no_missing_values: PASS
hire_dates_in_range: PASS
valid_departments: PASS
valid_job_levels: PASS


In [6]:

# Explore the fictional workforce composition

department_summary = (
    employees["initial_department"]
    .value_counts()
    .rename_axis("department")
    .reset_index(name="employees")
)

department_summary["percentage"] = (
    department_summary["employees"]
    / N_EMPLOYEES * 100
).round(2)

display(department_summary)


,department,employees,percentage
0,Engineering,696,34.80
1,Product,315,15.75
2,Operations,262,13.10
3,Data,216,10.80
4,Design,202,10.10
5,Finance,157,7.85
6,People,152,7.60


In [7]:

# Step 3A — Generate monthly observation dates

snapshot_dates = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="ME"
)

print("Number of monthly snapshots:", len(snapshot_dates))
print("First snapshot:", snapshot_dates.min().date())
print("Last snapshot:", snapshot_dates.max().date())


Number of monthly snapshots: 48
First snapshot: 2022-01-31
Last snapshot: 2025-12-31


In [8]:

# Step 3B — Create potential employee-month combinations

calendar = pd.DataFrame({
    "snapshot_date": snapshot_dates
})

employee_months = employees.merge(
    calendar,
    how="cross"
)

print("Potential employee-month combinations:",
      len(employee_months))

display(employee_months.head())


Potential employee-month combinations: 96000


,employee_id,hire_date,initial_department,initial_job_level,snapshot_date
0,E00001,2019-08-09,Engineering,L4,2022-01-31
1,E00001,2019-08-09,Engineering,L4,2022-02-28
2,E00001,2019-08-09,Engineering,L4,2022-03-31
3,E00001,2019-08-09,Engineering,L4,2022-04-30
4,E00001,2019-08-09,Engineering,L4,2022-05-31


In [9]:

# Retain observations on or after each employee's hire date

monthly_snapshots = employee_months.loc[
    employee_months["snapshot_date"]
    >= employee_months["hire_date"]
].copy()

monthly_snapshots = monthly_snapshots.reset_index(drop=True)

print("Rows after hire-date filtering:",
      len(monthly_snapshots))

display(monthly_snapshots.head(10))


Rows after hire-date filtering: 71512


,employee_id,hire_date,initial_department,initial_job_level,snapshot_date
0,E00001,2019-08-09,Engineering,L4,2022-01-31
1,E00001,2019-08-09,Engineering,L4,2022-02-28
2,E00001,2019-08-09,Engineering,L4,2022-03-31
3,E00001,2019-08-09,Engineering,L4,2022-04-30
4,E00001,2019-08-09,Engineering,L4,2022-05-31
5,E00001,2019-08-09,Engineering,L4,2022-06-30
6,E00001,2019-08-09,Engineering,L4,2022-07-31
7,E00001,2019-08-09,Engineering,L4,2022-08-31
8,E00001,2019-08-09,Engineering,L4,2022-09-30
9,E00001,2019-08-09,Engineering,L4,2022-10-31


In [10]:

# Step 3C — Calculate employee tenure

monthly_snapshots["tenure_years"] = (
    (
        monthly_snapshots["snapshot_date"]
        - monthly_snapshots["hire_date"]
    ).dt.days / 365.25
).round(2)

display(
    monthly_snapshots[
        ["employee_id", "snapshot_date",
         "hire_date", "tenure_years"]
    ].head(10)
)


,employee_id,snapshot_date,hire_date,tenure_years
0,E00001,2022-01-31,2019-08-09,2.48
1,E00001,2022-02-28,2019-08-09,2.56
2,E00001,2022-03-31,2019-08-09,2.64
3,E00001,2022-04-30,2019-08-09,2.72
4,E00001,2022-05-31,2019-08-09,2.81
5,E00001,2022-06-30,2019-08-09,2.89
6,E00001,2022-07-31,2019-08-09,2.98
7,E00001,2022-08-31,2019-08-09,3.06
8,E00001,2022-09-30,2019-08-09,3.14
9,E00001,2022-10-31,2019-08-09,3.23


In [11]:

# Step 3D — Assign starting monthly salary per employee

SALARY_BANDS = {
    "L1": (3000, 4500),
    "L2": (4500, 6500),
    "L3": (6500, 9000),
    "L4": (9000, 12500),
    "L5": (12500, 17000),
    "L6": (17000, 23000)
}

salary_rng = np.random.default_rng(seed=123)

employees["initial_monthly_salary"] = [
    round(
        salary_rng.uniform(*SALARY_BANDS[level]),
        2
    )
    for level in employees["initial_job_level"]
]

display(
    employees[
        ["employee_id", "initial_job_level",
         "initial_monthly_salary"]
    ].head()
)


,employee_id,initial_job_level,initial_monthly_salary
0,E00001,L4,11388.23
1,E00002,L3,6634.55
2,E00003,L2,4940.72
3,E00004,L4,9645.30
4,E00005,L1,3263.86


In [12]:

# Attach the starting salary to each monthly snapshot

monthly_snapshots = monthly_snapshots.merge(
    employees[
        ["employee_id", "initial_monthly_salary"]
    ],
    on="employee_id",
    how="left",
    validate="many_to_one"
)

monthly_snapshots = monthly_snapshots.rename(
    columns={
        "initial_department": "department",
        "initial_job_level": "job_level",
        "initial_monthly_salary": "monthly_salary"
    }
)

display(monthly_snapshots.head())


,employee_id,hire_date,department,job_level,snapshot_date,tenure_years,monthly_salary
0,E00001,2019-08-09,Engineering,L4,2022-01-31,2.48,11388.23
1,E00001,2019-08-09,Engineering,L4,2022-02-28,2.56,11388.23
2,E00001,2019-08-09,Engineering,L4,2022-03-31,2.64,11388.23
3,E00001,2019-08-09,Engineering,L4,2022-04-30,2.72,11388.23
4,E00001,2019-08-09,Engineering,L4,2022-05-31,2.81,11388.23


In [13]:

# Step 3E — Snapshot quality checks

snapshot_checks = {
    "unique_employee_month":
        not monthly_snapshots.duplicated(
            subset=["employee_id", "snapshot_date"]
        ).any(),

    "no_pre_hire_snapshots":
        (
            monthly_snapshots["snapshot_date"]
            >= monthly_snapshots["hire_date"]
        ).all(),

    "valid_departments":
        monthly_snapshots["department"].isin(
            DEPARTMENTS
        ).all(),

    "valid_job_levels":
        monthly_snapshots["job_level"].isin(
            JOB_LEVELS
        ).all(),

    "positive_salary":
        (
            monthly_snapshots["monthly_salary"] > 0
        ).all(),

    "nonnegative_tenure":
        (
            monthly_snapshots["tenure_years"] >= 0
        ).all(),

    "no_missing_values":
        not monthly_snapshots.isna().any().any()
}

for check, passed in snapshot_checks.items():
    print(
        f"{check}: {'PASS' if passed else 'FAIL'}"
    )

assert all(snapshot_checks.values()), (
    "Monthly snapshots failed validation."
)


unique_employee_month: PASS
no_pre_hire_snapshots: PASS
valid_departments: PASS
valid_job_levels: PASS
positive_salary: PASS
nonnegative_tenure: PASS
no_missing_values: PASS


In [14]:

print("Total snapshot rows:",
      len(monthly_snapshots))

print("Unique employees:",
      monthly_snapshots["employee_id"].nunique())

print("Distinct months:",
      monthly_snapshots["snapshot_date"].nunique())

print("\nDepartment distribution:")
display(
    monthly_snapshots["department"]
    .value_counts()
    .rename_axis("department")
    .reset_index(name="snapshot_rows")
)


Total snapshot rows: 71512
Unique employees: 2000
Distinct months: 48

Department distribution:


,department,snapshot_rows
0,Engineering,24756
1,Product,10954
2,Operations,9207
3,Data,7879
4,Design,7357
5,Finance,5767
6,People,5592


In [15]:

# Point-in-time headcount for one observation date
# NOTE: Valid active headcount only after exits are modeled.

example_date = pd.Timestamp("2024-12-31")

snapshot_view = monthly_snapshots.loc[
    monthly_snapshots["snapshot_date"] == example_date
]

display(
    snapshot_view.groupby("department")["employee_id"]
    .nunique()
    .reset_index(name="candidate_headcount")
)


,department,candidate_headcount
0,Data,194
1,Design,176
2,Engineering,622
3,Finance,143
4,Operations,226
5,People,137
6,Product,279


In [16]:

# Step 4A — Generate synthetic employment exits

exit_rng = np.random.default_rng(seed=456)

ANNUAL_EXIT_HAZARD = 0.12

# The employment spell can only be observed from the
# later of the employee's hire date or our study start.
risk_start = employees["hire_date"].clip(
    lower=START_DATE
)

# Number of observable days for each employee
observable_days = (
    END_DATE - risk_start
).dt.days.clip(lower=0)

# Convert annual hazard into probability of exiting
# during the observed period.
exit_probability = 1 - np.exp(
    -ANNUAL_EXIT_HAZARD
    * observable_days / 365.25
)

will_exit = (
    exit_rng.random(N_EMPLOYEES)
    < exit_probability.to_numpy()
)

print("Employees selected to exit:", will_exit.sum())
print("Share selected to exit:",
      round(will_exit.mean() * 100, 2), "%")


Employees selected to exit: 570
Share selected to exit: 28.5 %


In [17]:

# Step 4B — Assign exit dates

selected = employees.loc[will_exit].copy()

selected_risk_start = risk_start.loc[will_exit]
selected_days = observable_days.loc[will_exit].to_numpy()

# Exponential event times, conditioned on an
# event occurring during the observation window.
lambda_per_day = ANNUAL_EXIT_HAZARD / 365.25

u = exit_rng.random(len(selected))

event_days = (
    -np.log(
        1 - u * (
            1 - np.exp(-lambda_per_day * selected_days)
        )
    ) / lambda_per_day
)

# Convert continuous event times to whole days.
# Keep exits after risk_start and by END_DATE.
day_offsets = np.ceil(event_days).astype(int)

exit_dates = (
    selected_risk_start
    + pd.to_timedelta(day_offsets, unit="D")
)

selected["exit_date"] = exit_dates.to_numpy()

print("Generated exit dates:", len(selected))
print("Earliest exit:", selected["exit_date"].min())
print("Latest exit:", selected["exit_date"].max())


Generated exit dates: 570
Earliest exit: 2022-01-10 00:00:00
Latest exit: 2025-12-31 00:00:00


In [18]:

# Step 4C — Generate departure categories

selected["exit_type"] = exit_rng.choice(
    ["Voluntary", "Involuntary"],
    size=len(selected),
    p=[0.75, 0.25]
)

selected["exit_reason"] = [
    exit_rng.choice(
        ["Career Opportunity", "Personal Reasons",
         "Relocation", "Compensation"]
    )
    if exit_type == "Voluntary"
    else exit_rng.choice(
        ["Performance", "Restructuring",
         "Role Elimination"]
    )
    for exit_type in selected["exit_type"]
]

employment_exits = selected[
    ["employee_id", "exit_date",
     "exit_type", "exit_reason"]
].reset_index(drop=True)

display(employment_exits.head(10))


,employee_id,exit_date,exit_type,exit_reason
0,E00003,2024-11-06,Voluntary,Relocation
1,E00006,2025-05-16,Involuntary,Performance
2,E00007,2025-03-28,Involuntary,Restructuring
3,E00011,2024-01-17,Voluntary,Career Opportunity
4,E00017,2023-11-29,Voluntary,Compensation
5,E00020,2025-12-03,Voluntary,Personal Reasons
6,E00023,2023-01-28,Involuntary,Performance
7,E00024,2025-04-11,Voluntary,Personal Reasons
8,E00033,2024-07-02,Voluntary,Compensation
9,E00038,2025-11-01,Voluntary,Compensation


In [19]:

exit_summary = (
    employment_exits["exit_type"]
    .value_counts()
    .rename_axis("exit_type")
    .reset_index(name="departures")
)

display(exit_summary)


,exit_type,departures
0,Voluntary,442
1,Involuntary,128


In [20]:

# Step 4D — Merge exit dates into snapshots

snapshots_with_exits = monthly_snapshots.merge(
    employment_exits[
        ["employee_id", "exit_date", "exit_type"]
    ],
    on="employee_id",
    how="left",
    validate="many_to_one"
)

print("Rows before exit filtering:",
      len(snapshots_with_exits))


Rows before exit filtering: 71512


In [21]:

# Keep employees active at each month-end

active_mask = (
    snapshots_with_exits["exit_date"].isna()
    | (
        snapshots_with_exits["snapshot_date"]
        < snapshots_with_exits["exit_date"]
    )
)

active_snapshots = (
    snapshots_with_exits.loc[active_mask]
    .drop(columns=["exit_date", "exit_type"])
    .reset_index(drop=True)
)

print("Rows after exit filtering:",
      len(active_snapshots))

print("Removed post-exit snapshot rows:",
      len(snapshots_with_exits) - len(active_snapshots))


Rows after exit filtering: 59065
Removed post-exit snapshot rows: 12447


In [22]:

# Step 4E — Validate exit and snapshot integrity

exit_checks = {
    "unique_exit_employee":
        employment_exits["employee_id"].is_unique,

    "exit_employee_exists":
        employment_exits["employee_id"].isin(
            employees["employee_id"]
        ).all(),

    "valid_exit_dates":
        employment_exits["exit_date"].between(
            START_DATE,
            END_DATE
        ).all(),

    "valid_exit_types":
        employment_exits["exit_type"].isin(
            ["Voluntary", "Involuntary"]
        ).all(),

    "no_duplicate_active_snapshots":
        not active_snapshots.duplicated(
            ["employee_id", "snapshot_date"]
        ).any(),

    "no_pre_hire_active_snapshots":
        (
            active_snapshots["snapshot_date"]
            >= active_snapshots["hire_date"]
        ).all(),

    "no_missing_active_snapshot_values":
        not active_snapshots.isna().any().any()
}

for name, passed in exit_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

assert all(exit_checks.values()), (
    "Employment history validation failed."
)


unique_exit_employee: PASS
exit_employee_exists: PASS
valid_exit_dates: PASS
valid_exit_types: PASS
no_duplicate_active_snapshots: PASS
no_pre_hire_active_snapshots: PASS
no_missing_active_snapshot_values: PASS


In [23]:

# Step 4F — Cross-table reconciliation

exit_audit = employment_exits.merge(
    employees[["employee_id", "hire_date"]],
    on="employee_id",
    how="left",
    validate="one_to_one"
)

assert (
    exit_audit["exit_date"]
    > exit_audit["hire_date"]
).all(), "Found an exit on or before hire date."

post_exit_rows = active_snapshots.merge(
    employment_exits[["employee_id", "exit_date"]],
    on="employee_id",
    how="left",
    validate="many_to_one"
)

invalid_post_exit = post_exit_rows[
    post_exit_rows["exit_date"].notna()
    & (
        post_exit_rows["snapshot_date"]
        >= post_exit_rows["exit_date"]
    )
]

print("Invalid post-exit snapshots:",
      len(invalid_post_exit))

assert len(invalid_post_exit) == 0


Invalid post-exit snapshots: 0


In [24]:

report_date = pd.Timestamp("2024-12-31")

headcount_by_department = (
    active_snapshots.loc[
        active_snapshots["snapshot_date"] == report_date
    ]
    .groupby("department")["employee_id"]
    .nunique()
    .reset_index(name="active_headcount")
    .sort_values(
        "active_headcount",
        ascending=False
    )
)

display(headcount_by_department)

print(
    "Total active headcount:",
    headcount_by_department["active_headcount"].sum()
)


,department,active_headcount
2,Engineering,477
6,Product,225
4,Operations,172
1,Design,139
0,Data,136
3,Finance,115
5,People,108


Total active headcount: 1372


In [25]:

from pathlib import Path

# Organize datasets inside the Colab workspace
output_dir = Path("data/synthetic")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir.resolve())


Output directory: /content/data/synthetic


In [26]:

# Export the validated synthetic datasets

employees.to_csv(
    output_dir / "employees.csv",
    index=False
)

employment_exits.to_csv(
    output_dir / "employment_exits.csv",
    index=False
)

active_snapshots.to_csv(
    output_dir / "monthly_snapshots.csv",
    index=False
)

print("Export completed.")
for file in sorted(output_dir.glob("*.csv")):
    print(file.name, "-", round(file.stat().st_size / 1024, 1), "KB")


Export completed.
employees.csv - 75.2 KB
employment_exits.csv - 24.2 KB
monthly_snapshots.csv - 3135.5 KB


In [27]:

# Read the exported CSVs

employees_check = pd.read_csv(
    output_dir / "employees.csv"
)

exits_check = pd.read_csv(
    output_dir / "employment_exits.csv"
)

snapshots_check = pd.read_csv(
    output_dir / "monthly_snapshots.csv"
)

assert len(employees_check) == len(employees)
assert len(exits_check) == len(employment_exits)
assert len(snapshots_check) == len(active_snapshots)

assert employees_check["employee_id"].is_unique
assert exits_check["employee_id"].is_unique
assert not snapshots_check.duplicated(
    ["employee_id", "snapshot_date"]
).any()

print("PASS: Exported row counts and primary keys validated.")


PASS: Exported row counts and primary keys validated.


In [28]:

# Remove redundant column if present
active_snapshots = active_snapshots.drop(
    columns=["initial_monthly_salary"],
    errors="ignore"
)

# Re-export the clean snapshot table
active_snapshots.to_csv(
    output_dir / "monthly_snapshots.csv",
    index=False
)

print(active_snapshots.columns.tolist())


['employee_id', 'hire_date', 'department', 'job_level', 'snapshot_date', 'tenure_years', 'monthly_salary']


In [29]:

from google.colab import files

files.download("data/synthetic/employees.csv")
files.download("data/synthetic/employment_exits.csv")
files.download("data/synthetic/monthly_snapshots.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>